# F5-TTS Fine-Tuned for Vietnamese

## What is F5-TTS?

[F5-TTS](https://github.com/SWivid/F5-TTS) is a **flow-matching based text-to-speech** system developed by SWivid. Unlike autoregressive TTS models (e.g., XTTS), F5-TTS uses a non-autoregressive architecture built on flow matching and a Diffusion Transformer (DiT), which makes it significantly **faster at inference time**.

### Why a Vietnamese Fine-Tune?

The base F5-TTS model is primarily trained on English data and does not produce good results for Vietnamese. However, the community has created excellent fine-tuned checkpoints specifically for Vietnamese:

| Model | Training Data | Notes |
|-------|--------------|-------|
| [hynt/F5-TTS-Vietnamese-ViVoice](https://huggingface.co/hynt/F5-TTS-Vietnamese-ViVoice) | ~1000 hours | **Best quality**, recommended |
| [hynt/F5-TTS-Vietnamese-100h](https://huggingface.co/hynt/F5-TTS-Vietnamese-100h) | ~100 hours | Smaller, lighter alternative |

### Key Capabilities

- **Zero-shot voice cloning**: Provide just 3-10 seconds of reference audio and the model reproduces that voice
- **High quality Vietnamese speech**: Tones, pronunciation, and naturalness are well-captured
- **Fast inference**: Flow matching is non-autoregressive, much faster than XTTS-based approaches

### How It Works

The F5-TTS approach is simple:
1. Provide a **reference audio** clip (3-10 seconds)
2. Provide the **reference text** (what is spoken in the reference audio)
3. Provide the **target text** (what you want the model to say)
4. The model generates speech of the target text **in the voice of the reference audio**

## 1. Installation

Install the `f5-tts` package and fix a protobuf compatibility issue that can occur on Kaggle.

In [ ]:
!pip install -q f5-tts

In [ ]:
!pip install -q --force-reinstall 'protobuf>=5.26.1'

## 2. Download the Vietnamese Fine-Tuned Checkpoint

We download the best checkpoint: **hynt/F5-TTS-Vietnamese-ViVoice** (trained on ~1000 hours of Vietnamese speech data).

The model is distributed as a `.safetensors` file on HuggingFace.

In [ ]:
from huggingface_hub import hf_hub_download

# Download the Vietnamese fine-tuned checkpoint
ckpt_path = hf_hub_download(
    repo_id="hynt/F5-TTS-Vietnamese-ViVoice",
    filename="model_last.safetensors",
)

print(f"Checkpoint downloaded to: {ckpt_path}")

## 3. Load the Model

We use the `F5TTS` class from the `f5_tts.api` module. We point it to our downloaded Vietnamese checkpoint.

In [ ]:
from f5_tts.api import F5TTS

# Initialize the model with the Vietnamese fine-tuned checkpoint
f5tts = F5TTS(
    model_type="F5-TTS",
    ckpt_file=ckpt_path,
)

print("Model loaded successfully!")

## 4. Prepare Reference Audio

F5-TTS requires a short reference audio clip (3-10 seconds) along with its transcript. The model will clone the voice from this reference.

For this demo, we create a simple test reference. In practice, you would use a real Vietnamese speech recording.

In [ ]:
import os

# --- Option A: Use your own reference audio ---
# Place a .wav file (3-10 seconds, mono, 16kHz or higher) and set the path:
# ref_audio = "/path/to/your/reference.wav"
# ref_text = "Transcript of what is spoken in the reference audio"

# --- Option B: Use F5-TTS built-in test reference ---
# The f5-tts package ships with a test reference audio.
# We can find it in the package directory.
import f5_tts

package_dir = os.path.dirname(f5_tts.__file__)
test_ref_dir = os.path.join(package_dir, "infer", "examples", "basic")

# List available reference files
if os.path.exists(test_ref_dir):
    print("Available test references:")
    for f in sorted(os.listdir(test_ref_dir)):
        print(f"  {f}")
else:
    print(f"Test reference directory not found at {test_ref_dir}")
    print("You will need to provide your own reference audio.")

In [ ]:
# Set up the reference audio path and text
# Adjust these to match your own reference if needed

ref_audio = os.path.join(test_ref_dir, "basic_ref_en.wav")  # adjust filename as needed
ref_text = "Some call me nature, others call me mother nature."  # transcript of ref audio

# Verify the file exists
if os.path.exists(ref_audio):
    import soundfile as sf
    data, sr = sf.read(ref_audio)
    duration = len(data) / sr
    print(f"Reference audio: {ref_audio}")
    print(f"  Sample rate: {sr} Hz")
    print(f"  Duration: {duration:.2f} seconds")
else:
    print(f"File not found: {ref_audio}")
    print("Please update ref_audio path to point to a valid .wav file.")

## 5. Basic Speech Generation

Now we generate Vietnamese speech. The workflow is:
- `ref_file`: path to the reference audio
- `ref_text`: what is said in the reference audio
- `gen_text`: the Vietnamese text we want the model to speak

The model will produce speech in the same voice as the reference audio.

In [ ]:
# Generate Vietnamese speech
gen_text = "Xin chào, tôi là trợ lý ảo. Tôi có thể giúp gì cho bạn hôm nay?"

wav, sr, _ = f5tts.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=gen_text,
)

print(f"Generated audio: {len(wav)/sr:.2f} seconds at {sr} Hz")

In [ ]:
# Play the generated audio in the notebook
from IPython.display import Audio, display

print(f"Text: {gen_text}")
display(Audio(wav, rate=sr))

In [ ]:
# Save the generated audio to a file
import soundfile as sf

output_path = "output_vietnamese.wav"
sf.write(output_path, wav, sr)
print(f"Saved to {output_path}")

## 6. Multiple Vietnamese Text Examples

Let's generate several Vietnamese sentences to hear how the model handles different content.

In [ ]:
vietnamese_texts = [
    "Việt Nam là một đất nước xinh đẹp với nhiều danh lam thắng cảnh nổi tiếng.",
    "Hà Nội là thủ đô của Việt Nam, nổi tiếng với phố cổ và ẩm thực đường phố.",
    "Trí tuệ nhân tạo đang thay đổi thế giới theo những cách chúng ta chưa từng tưởng tượng.",
    "Chúc bạn một ngày tốt lành và nhiều niềm vui.",
    "Công nghệ chuyển đổi văn bản thành giọng nói ngày càng phát triển vượt bậc.",
]

for i, text in enumerate(vietnamese_texts, 1):
    print(f"\n--- Example {i} ---")
    print(f"Text: {text}")

    wav, sr, _ = f5tts.infer(
        ref_file=ref_audio,
        ref_text=ref_text,
        gen_text=text,
    )

    display(Audio(wav, rate=sr))

    # Save each example
    sf.write(f"output_vn_example_{i}.wav", wav, sr)
    print(f"Saved to output_vn_example_{i}.wav")

## 7. Architecture: How F5-TTS Works

### Flow Matching TTS

F5-TTS uses **Conditional Flow Matching (CFM)** rather than the traditional autoregressive approach. Here is a simplified view of the architecture:

```
                    F5-TTS Architecture
                    ===================

  Reference Audio ──> Mel Spectrogram ──┐
                                        │
  Reference Text  ──> Text Encoder ─────┤
                                        ├──> Diffusion Transformer (DiT)
  Target Text     ──> Text Encoder ─────┤         │
                                        │         │ Flow Matching
  Gaussian Noise  ─────────────────────>─┘         │ (iterative denoising)
                                                   │
                                                   v
                                        Generated Mel Spectrogram
                                                   │
                                                   v
                                              Vocoder (Vocos)
                                                   │
                                                   v
                                          Output Waveform (audio)
```

### Key Concepts

1. **Flow Matching**: Instead of generating tokens one by one (autoregressive), the model learns a continuous flow from noise to speech. This is done in mel-spectrogram space and allows parallel generation.

2. **Diffusion Transformer (DiT)**: The core architecture is a transformer that processes the noisy mel spectrogram conditioned on text. It iteratively refines the spectrogram over multiple denoising steps.

3. **Non-Autoregressive**: The entire spectrogram is generated at once (with iterative refinement), not token-by-token. This makes it significantly faster than autoregressive models.

4. **Vocos Vocoder**: The generated mel spectrogram is converted to a waveform using the Vocos vocoder.

### Comparison: Flow Matching vs Autoregressive (XTTS)

| Feature | F5-TTS (Flow Matching) | XTTS (Autoregressive) |
|---------|----------------------|----------------------|
| Generation | Non-autoregressive (parallel) | Autoregressive (sequential) |
| Speed | Faster | Slower |
| Architecture | DiT + Flow Matching | GPT-like + HiFi-GAN |
| Voice Cloning | 3-10s reference | 6-30s reference |
| Quality | High (with fine-tune) | High |

## 8. Advanced: Adjusting Generation Parameters

F5-TTS allows some control over the generation process.

In [ ]:
# Generate with custom parameters
gen_text_adv = "Đây là một ví dụ về giọng nói tiếng Việt được tạo bởi trí tuệ nhân tạo."

wav_adv, sr_adv, _ = f5tts.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=gen_text_adv,
    nfe_step=32,        # Number of flow matching steps (default: 32, higher = better quality but slower)
    speed=1.0,          # Speed factor (1.0 = normal, <1 = slower, >1 = faster)
)

print(f"Text: {gen_text_adv}")
display(Audio(wav_adv, rate=sr_adv))

## 9. Key Takeaways

1. **High Quality Vietnamese TTS**: The `hynt/F5-TTS-Vietnamese-ViVoice` model, trained on ~1000 hours of Vietnamese data, produces natural-sounding Vietnamese speech with proper tones and pronunciation.

2. **Zero-Shot Voice Cloning**: Just 3-10 seconds of reference audio is enough to clone a voice. No speaker-specific fine-tuning is needed.

3. **Faster Than viXTTS**: Because F5-TTS uses flow matching (non-autoregressive), it is significantly faster at inference time compared to XTTS-based approaches like viXTTS, while achieving comparable or better quality.

4. **Simple API**: The `f5_tts.api.F5TTS` class provides a clean interface: load a checkpoint, call `infer()` with reference audio and target text.

5. **Community-Driven**: The Vietnamese fine-tunes are community contributions, demonstrating how open-source TTS models can be adapted for any language with sufficient training data.

### Resources

- [F5-TTS GitHub](https://github.com/SWivid/F5-TTS)
- [F5-TTS Paper](https://arxiv.org/abs/2410.06885)
- [Vietnamese Checkpoint (ViVoice)](https://huggingface.co/hynt/F5-TTS-Vietnamese-ViVoice)
- [Vietnamese Checkpoint (100h)](https://huggingface.co/hynt/F5-TTS-Vietnamese-100h)